In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path
from zipfile import ZipFile, ZIP_STORED

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
import numpy as np
from contextlib import ExitStack
from io import BytesIO
from PIL import Image

In [ ]:
DRIVE = Path('/content/drive/MyDrive/WasteScope')
ARCHIVE_DIR = DRIVE / 'raw_archives'
TASKS = {
    'detection': ARCHIVE_DIR / 'detection_task.zip',
    'tracking': ARCHIVE_DIR / 'tracking_task.zip',
    'segmentation': ARCHIVE_DIR / 'segmentation_task.zip'
}
CLASSES = [
    'container_yellow', 'container_green', 'container_default',
    'container_blue', 'container_oil', 'container_battery',
    'container_biodegradable'
]

OUTPUT = DRIVE / 'prepared' / 'detection_7class_v2'
MANIFEST = DRIVE / 'manifests' / 'video_split_7class_v2.json'
SEED = 42
RATIOS = np.array([0.70, 0.15, 0.15])
N_TRIALS = 1000
SPLIT_NAMES = ('train', 'val', 'test')

In [ ]:
with ZipFile(TASKS['segmentation']) as archive:
    json_name = next(name for name in archive.namelist() if Path(name).name == 'labels.json')
    coco = json.loads(archive.read(json_name).decode('utf-8-sig'))
    
categories = {item['id']: item['name'] for item in coco['categories']}
overflow_id = next(cid for cid, name in categories.items() if name.lower() == 'overflow')
first_container = min(cid for cid, name in categories.items() if name.startswith('container_'))

In [ ]:
DETECTION_MAP = {
    cid - 1: CLASSES.index(name) for cid, name in categories.items() if name in CLASSES
}
TRACKING_MAP = {
    cid - first_container: CLASSES.index(name)
    for cid, name in categories.items() if name in CLASSES
}

DETECTION_SKIP = {cid - 1 for cid, name in categories.items() if name not in CLASSES}
TRACKING_SKIP = {
    cid - first_container for cid, name in categories.items()
    if name.startswith('container_') and name not in CLASSES
}

In [7]:
images, labels = {}, {}

for task, path in TASKS.items():
    with ZipFile(path) as archive:
        image_names = sorted(name for name in archive.namelist() if name.lower().endswith('.png'))
        label_names = sorted(name for name in archive.namelist() if name.lower().endswith('.txt') and Path(name).parent.name == 'labels')
        
    images[task] = {Path(name).stem: name for name in image_names}
    labels[task] = {Path(name).stem: name for name in label_names}

distribution = pd.DataFrame({
    task: Counter(stem.split('_')[0] for stem in files)
    for task, files in images.items()
}).fillna(0).astype(int).sort_index()

distribution.index.name = 'Video'

In [8]:
detection_labels = {}

for task, mapping, excluded, fields_per_row in (
    ('detection', DETECTION_MAP, DETECTION_SKIP, 5),
    ('tracking', TRACKING_MAP, TRACKING_SKIP, 7)
):
    instances = defaultdict(Counter)
    background = Counter()
    with ZipFile(TASKS[task]) as archive:
        for stem, member in tqdm(sorted(labels[task].items())):
            video = stem.split("_")[0]
            converted = []
            for line in archive.read(member).decode("utf-8-sig").splitlines():
                fields = line.split()
                if not fields:
                    continue
                if len(fields) != fields_per_row:
                    raise ValueError(f'Invalid label format: {member}')
                
                raw_id = int(fields[0])
                if raw_id in excluded:
                    continue
                
                cid = mapping[raw_id]
                instances[video][cid] += 1
                converted.append(' '.join([str(cid), *fields[1:]]))
            background[video] += not converted
            if task == 'detection':
                detection_labels[stem] = '\n'.join(converted) + ('\n' if converted else '')
    
     
    for cid, name in enumerate(CLASSES):
        distribution[f'{task}:{name}'] = [instances[v][cid] for v in distribution.index]
    distribution[f'{task}:background'] = [background[v] for v in distribution.index]

  0%|          | 0/36478 [00:00<?, ?it/s]

  0%|          | 0/10022 [00:00<?, ?it/s]